# Numpy  
vectorized operations implemented in C
*(>10x faster)*

In [17]:
import numpy as np
#import numpy and give the variable name np now we can use np to access its functions
print(np.__version__)
#check version of numpy

2.5.0


In [18]:
#without numpy
my_list = [1,2,3,4]
my_list = my_list * 2
print(my_list)
#multiplying a list by 2 will concatanate the list with itself while what we want is to multiply each element by 2

[1, 2, 3, 4, 1, 2, 3, 4]


In [19]:
#now with numpy
arr = np.array([1,2,3,4])
arr = arr * 2
print(arr)
print(type(arr))

[2 4 6 8]
<class 'numpy.ndarray'>


## Multidimensional arrays & indexing

In [20]:
# now arrays have dimensions and shape, dimensions is obviously the number of dimensions or axis it has, while shape is the number
# of elements in each dimension
array_0d = np.array('A') #think single point
print (array_0d.ndim) 
print (array_0d.shape)
#while a 1d:
array_1d = np.array(['A','B','C']) #similar to a line
print(array_1d.ndim)
print(array_1d.shape)
#while a 2d:
array_2d = np.array([['A','B','C'],
                     ['D','E','F']])
print(array_2d.ndim)
print(array_2d.shape)
#while a 3d:
array_3d = np.array([[['A','B','C'],['D','E','F']],
                     [['G','H','I'],['J','K','L']]])
print(array_3d.ndim)
print(array_3d.shape)

0
()
1
(3,)
2
(2, 3)
3
(2, 2, 3)


In [21]:
#however np arrays have to be of the same data type as such np will automatically implicitly upcast all data types to the lowest common denominator that can handle all the data
upcast_array = np.array([1,2.5,5])
print(upcast_array) # notice the dot implying float

[1.  2.5 5. ]


In [22]:
#the axis/dimensions also have to be homogeneous in size so they all have to have the same number of elements in each dimension, otherwise it will be a ragged array and will be treated as an object array
#ragged_array = np.array([[1,2,3],[4,5]])
#print(ragged_array)#Error

In [23]:
#finally accessing elements in a np array is similar to lists where we would use something called chain indexing
#ex:
print(array_2d[0][2])
#however this is not the most efficient or fastest way, as we also have what is called multi-dimensional indexing which is more efficient and faster
print(array_2d[0,2])

C
C


## slicing and dimension collapse

In [24]:
array = np.array([[1,2,3,4]
                  ,[5,6,7,8]
                  ,[9,10,11,12]
                  ,[13,14,15,16]])
#how slicing follows this syntax array[start:end(exclusive):step]
#for example say we want the first row
print(array[0:1])
print()#for clarity
#what if we want the first and second
print(array[0:2])
print()
#what if we want first and third (step of 2)
print(array[0:3:2])
print()
#we can also do the same with columns
print(array[:,0])#notice the first axis has a ':' which numpy interprates as grab all, and it is actually what it autofills if we stop at the first axis
#example print(array[0:2])  actually expands to print(array[0:2,:])
print()
#now if we want the first and third col
print(array[:,0::2])#ommiting the end to mean grab till end


[[1 2 3 4]]

[[1 2 3 4]
 [5 6 7 8]]

[[ 1  2  3  4]
 [ 9 10 11 12]]

[ 1  5  9 13]

[[ 1  3]
 [ 5  7]
 [ 9 11]
 [13 15]]


now notice when we did `print(array[:,0])`  
the output was `[ 1  5  9 13]`
but when when we did `print(array[:,0::2])`
the output became  
```text
[[ 1  3]
[ 5  7]
[ 9 11]
[13 15]]
```  
now why is this important? notice the different shape of each output, yes ofcourse the number of elements is different, duh  
but i am referring to the number of brackets as well, notice that one has one bracket (looks like a 1d vector), while
the other has 2, aka it is a matrix, now we can even prove this by printing their shapes

In [25]:
print(array[:,0].shape)
print(array[:,0::2].shape)

(4,)
(4, 2)


notice how the second shows `(4, 2)` while the first shows `(4,)` not `(4,1)` but `(4,)` that shows that it is indeed collapsed into a 1d vector  
this is because numpy when slicing if we choose a specific index instead of a subsection(even if it had a size of 1) collapses that axis down  
so `array[:,0]` isnt a 2d matrix with 1 column and 4 rows but actually a simply 1d vector  
had we wanted to get the 2d we would use `array[:,0:1]`

In [26]:
print(array[:,0:1].shape)

(4, 1)


this is __very__ important for dimensional mismatch later on, especially with DL and weights.

## Arithmetic  
there are many different kinds of arithmetic supported by numpy but they all depend on how numpy optimizes vector operations
so while a regular scalar operation would, when adding two arrays, load from MEM a[0] then load from MEM b[0] then add then store in c[0] then go to a[1]..etc  
numpy uses many optimizations to cut this down by more than 10x
first it uses the fact that CPUs haver caches which are 100s of times faster to query than the memory (L1,L2,L3 in increasing order of size and decreasing speed), using said caches would be great but regularly python stores each element in a list as a pointer to an object
since elements and usually non homogenous (can mix diff. types), but since numpy enforces strict type homogeniety, now we can load all elements next to each other and load a bunch together into cache, absolutely minimizing cache misses (which makes us query the memory faaaar less)  
second, it makes use of special instruction set extensions called AVX2 and AVX-512(for intel processors, AMD has similar extenstions), what it basically does is make use of a hardware component, that is expanding the 64/128 bit registers in place to accomodate 256 and 512 bits respectively (called YMM and ZMM registers), and a software component, that is new Floating point instructions specifically made to use the new larger registers to make use of them to calculate many floating point operations at once, think for example if a single element was 32 bits, a single YMM register could accomodate `256/32 = 38` elements if we were to use these to apply a single operation to all of them at the same time (SIMD) we would cut running time by 8x
thirdly, it also uses many other optimizations such as highly optimized external math libraries (BLAS, intel MKL, etc) that allow multithreading (remember python GIL strictly enforces one thread to run python code at a time, bottlenecking multicore usage), but we won't go into much details on the other optimizations for now.

### Scalar arithmetic

scalar as in adding a singular element to a whole array, this uses a concept called broadcasting that we will go into in detail later, for now it is enough to know that it means *stretching* the element/smaller array to the size of the bigger one so operations are viable.
so when lets say we want to add `[1,2,3] + 3`, regular python lists would throw an error while numpy would first duplicate the element and add it to all elements in our bigger array, `[1,2,3] + [3,3,3]` and making use of our previously explained optimization cut down on running time significantly without a single for loop in sight! 

In [27]:
arr = np.array([1,2,3])
print(arr + 3)

[4 5 6]


### Vectorized math functions

simply an extension of the previous idea, where now we can use built-in numpy math functions and constants on all array elements at once!

In [28]:
print(np.sqrt(arr))

tmp_float_arr = np.array([1.3,2.7,3.9])
print(np.round(tmp_float_arr))

print(np.pi)

#calculate circle areas for radii
radii = np.array([1,5,20,100])
print(np.pi * radii ** 2) #done in 2 cycles (not accounting for time to load in cache) instead of 4 *4 * 4 ~= 256

[1.         1.41421356 1.73205081]
[1. 3. 4.]
3.141592653589793
[3.14159265e+00 7.85398163e+01 1.25663706e+03 3.14159265e+04]


### Element-wise arithmetic  

doing arithmetic between two arrays element-by-element (again utilizes all our previous optimizations) and follows broadcasting and shape matching rules, we will discuss those later.

In [29]:
arr1 = np.array([[1,2,3],[4,5,6],[7,8,9]])
arr2 = np.array([[9,8,7],[6,5,4],[3,2,1]])
print(arr1 + arr2) #can also use np.add(arr1,arr2) but both are congruent EXCEPT when you want to use np.add options like out,..,etc

[[10 10 10]
 [10 10 10]
 [10 10 10]]


### Comparison operators

Now, these are an actually fascination usage of numpy think if we wanted to check an array of scores for passing scores, and return a true/false depending on each score or maybe even assign a new values for failing scores, regular python would do something like this:  
```python
scores = [30,60,78,100]
for x in scores:
    if (x > 60):
        print(x) #/true
    else:
        print(0) #/false
```
however in numpy this is all that takes

In [30]:
scores = np.array([30,60,78,100])
scores[scores < 60] = 0
print(scores)

[  0  60  78 100]


and ofcourse the running time is magnitudes faster

this syntax `arr[arr > x]` is actually called a boolean mask where numpy does its optimized SIMD operations to calculate a boolean mask for the array to evaluate the operation on each element returning something that looks like `[true, false, false, false]` it then applies that mask likes a physical shield to the cache only allowing the 0 to pass to locations that don't have a false to physically protect it

## Broadcasting